# Recherche Stratégie #5 : Quality + Low Volatility Factor Tilt

**EPIC #1409 — Curriculum Trading V3, Issue #35 Stratégie 5**  
**Branche** : `feature/quality-lowvol-research`  
**Date** : 2026-05-26 | **Auteur** : myia-po-2024

---

## Navigation

| # | Stratégie | Verdict | PR |
|---|-----------|---------|-----|
| 1 | Dual Momentum (L2) | NO BEATS | #1575 |
| 2 | Risk Parity | NO BEATS | #1578 |
| 3 | Trend Following (L1 TSMOM) | NO BEATS | #1574 |
| 4 | VRP Put-Write | NO BEATS | #1579 |
| **5** | **Quality + LowVol** | **En cours** | **Ce notebook** |

## Objectifs d'apprentissage

1. Comprendre les facteurs Quality (qualité des bénéfices) et Low Volatility (anomalie low-vol)
2. Implémenter un scoring combiné qualité + low-vol sur le panier multi-actifs
3. Comparer un portefeuille factor-tilted vs equal-weight B&H via walk-forward 5-fold × 4 seeds

## Hypothèse

Les actifs de haute qualité (momentum positif, Sharpe élevé) et faible volatilité surperforment
en termes de ratio risque-rendement. Le tilt factoriel mensuel pourrait générer un Sharpe > B&H.

## Prérequis

- Python 3.10+, pandas, numpy, yfinance
- Panier anti-bias 25 symboles (L1/L2/L3 data)

**Durée estimée** : 15 min

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Config
PANIER_CSV = Path("c:/dev/CoursIA/MyIA.AI.Notebooks/QuantConnect/datasets/panier/panier_close_all.csv")
LOOKBACK = 252           # 1y lookback for factor scoring
VOL_LOOKBACK = 63        # 3m volatility
REBAL_FREQ = 21          # monthly rebalance
N_LONG = 8               # top N assets to hold
TX_COST = 0.001          # 10bps transaction cost per trade
SEEDS = [0, 1, 7, 42]
N_SPLITS = 5
FEE_BPS = 10

# Load panier
df = pd.read_csv(PANIER_CSV, index_col=0, parse_dates=True)
df.index = pd.DatetimeIndex(df.index)
df = df.sort_index()
# Exclude VIX (volatility index, no price return)
exclude = ['VIX']
cols = [c for c in df.columns if c not in exclude]
prices = df[cols].ffill().dropna(how='all')
# Keep only business days for fair comparison (crypto has 365d, equities only weekdays)
prices = prices[prices.index.dayofweek < 5]
print(f"Panier: {prices.shape[1]} symbols, {prices.shape[0]} jours (business days only)")
print(f"Periode: {prices.index.min().date()} -> {prices.index.max().date()}")
print(f"Symbols: {list(prices.columns)}")

Panier: 25 symbols, 2957 jours (business days only)
Periode: 2015-01-01 -> 2026-05-01
Symbols: ['SPY', 'RSP', 'IWM', 'XLF', 'XLK', 'XLV', 'XLY', 'XLP', 'XLI', 'XLU', 'XLB', 'XLRE', 'XLC', 'TLT', 'IEF', 'SHY', 'GLD', 'USO', 'DBA', 'EFA', 'EEM', 'BTC-USD', 'ETH-USD', 'LTC-USD', 'XRP-USD']


## 1. Construction des scores factoriels

On combine deux signaux :
- **Quality** : rendement annualisé sur la période de lookback (proxy pour la qualité)
- **Low Volatility** : volatilité inverse (1/vol) — préférer les actifs moins volatils

Le score composite normalise chaque facteur (z-score cross-sectionnel) puis les combine 50/50.

In [2]:
def compute_factor_scores(prices: pd.DataFrame, lookback: int = 252, vol_lb: int = 63) -> pd.DataFrame:
    """Compute monthly quality + low-vol factor scores.
    
    Quality = annualized return over lookback (momentum as quality proxy).
    Low Vol = inverse of realized vol over vol_lb.
    Score = 0.5 * z(quality) + 0.5 * z(low_vol)
    """
    ret = prices.pct_change()
    # Quality: rolling annualized return
    quality = ret.rolling(lookback, min_periods=int(lookback * 0.8)).mean() * 252
    # Low vol: inverse of rolling vol
    vol = ret.rolling(vol_lb, min_periods=int(vol_lb * 0.8)).std() * 252**0.5
    low_vol = 1.0 / vol.replace(0, np.nan)
    
    # Resample to monthly (end of month)
    quality_m = quality.resample('ME').last()
    low_vol_m = low_vol.resample('ME').last()
    
    # Cross-sectional z-score each month
    def zscore_cross(row):
        valid = row.dropna()
        if len(valid) < 3:
            return pd.Series(np.nan, index=row.index)
        mu, sd = valid.mean(), valid.std()
        if sd < 1e-12:
            return pd.Series(0.0, index=row.index)
        return (row - mu) / sd
    
    q_z = quality_m.apply(zscore_cross, axis=1)
    lv_z = low_vol_m.apply(zscore_cross, axis=1)
    
    composite = 0.5 * q_z + 0.5 * lv_z
    return composite.dropna(how='all')

scores = compute_factor_scores(prices)
# Check coverage
valid_per_row = scores.notna().sum(axis=1)
print(f"Factor scores: {scores.shape[0]} mois x {scores.shape[1]} symboles")
print(f"Valid scores per row: min={valid_per_row.min()}, median={valid_per_row.median()}, max={valid_per_row.max()}")
print(f"\nDernier mois top-5:")
last = scores.iloc[-1].dropna().sort_values(ascending=False)
for sym, val in last.head(5).items():
    print(f"  {sym:8s}: score = {val:+.3f}")
print(f"\nDernier mois bottom-3:")
for sym, val in last.tail(3).items():
    print(f"  {sym:8s}: score = {val:+.3f}")

Factor scores: 128 mois x 25 symboles
Valid scores per row: min=21, median=25.0, max=25

Dernier mois top-5:
  SHY     : score = +2.061
  USO     : score = +1.133
  GLD     : score = +0.342
  EEM     : score = +0.279
  IEF     : score = +0.264

Dernier mois bottom-3:
  BTC-USD : score = -0.954
  XRP-USD : score = -1.335
  LTC-USD : score = -1.368


## 2. Backtest factor-tilted portfolio

A chaque fin de mois, on sélectionne les `N_LONG` meilleurs actifs selon le score composite.
Le portefeuille est equal-weight parmi les sélectionnés. On compare à B&H equal-weight tous les actifs.
Coûts de transaction : `TX_COST` par trade (rotation mensuelle).

In [3]:
def run_factor_tilt_backtest(prices: pd.DataFrame, scores: pd.DataFrame, n_long: int = 8,
                            tx_cost: float = 0.001) -> pd.DataFrame:
    """Run monthly factor-tilted backtest.
    
    Each month: select top n_long by composite score, equal-weight.
    Returns net of tx costs on position changes.
    """
    ret = prices.pct_change()
    # Monthly returns (next month after signal)
    monthly_ret = ret.resample('ME').apply(lambda x: (1+x).prod()-1)
    
    signal_dates = scores.index
    ret_dates = monthly_ret.index
    
    factor_ret = []
    bh_ret = []
    dates = []
    prev_holdings = None
    
    for i in range(len(signal_dates) - 1):
        sig_date = signal_dates[i]
        future_rets = ret_dates[ret_dates > sig_date]
        if len(future_rets) == 0:
            break
        ret_date = future_rets[0]
        if ret_date not in monthly_ret.index:
            continue
        
        row_scores = scores.loc[sig_date].dropna()
        if len(row_scores) < n_long:
            continue
        selected = row_scores.nlargest(n_long).index.tolist()
        
        next_ret = monthly_ret.loc[ret_date]
        valid_selected = [s for s in selected if s in next_ret.index and not np.isnan(next_ret[s])]
        if len(valid_selected) < 3:
            continue
        port_ret = next_ret[valid_selected].mean()
        
        if prev_holdings is not None:
            turnover = len(set(valid_selected) - set(prev_holdings)) + len(set(prev_holdings) - set(valid_selected))
            turnover_frac = turnover / (2 * n_long)
            port_ret -= turnover_frac * tx_cost
        else:
            port_ret -= tx_cost
        
        all_valid = next_ret.dropna()
        bh = all_valid.mean()
        
        factor_ret.append(port_ret)
        bh_ret.append(bh)
        dates.append(ret_date)
        prev_holdings = valid_selected
    
    result = pd.DataFrame({
        'factor': factor_ret,
        'bh': bh_ret
    }, index=pd.DatetimeIndex(dates))
    return result

bt = run_factor_tilt_backtest(prices, scores, n_long=N_LONG, tx_cost=TX_COST)

def sharpe_ann(returns):
    if len(returns) < 3:
        return float('nan')
    mu = returns.mean()
    sigma = returns.std()
    if sigma < 1e-12:
        return float('nan')
    return (mu / sigma) * 12**0.5

factor_sharpe = sharpe_ann(bt['factor'])
bh_sharpe = sharpe_ann(bt['bh'])

if len(bt) > 0:
    factor_cagr = (1 + bt['factor']).prod() ** (12/len(bt)) - 1
    bh_cagr = (1 + bt['bh']).prod() ** (12/len(bt)) - 1
else:
    factor_cagr = bh_cagr = float('nan')

print(f"{'='*70}")
print(f"Backtest Quality+LowVol: {len(bt)} mois, top-{N_LONG}, tx={TX_COST*10000:.0f}bps")
print(f"{'='*70}")
print(f"Factor tilt: Sharpe={factor_sharpe:.3f}, CAGR={factor_cagr:.2%}")
print(f"B&H equal:   Sharpe={bh_sharpe:.3f}, CAGR={bh_cagr:.2%}")
print(f"Delta Sharpe: {factor_sharpe - bh_sharpe:+.3f}")
if len(bt) > 0:
    print(f"\nCumulative: Factor={(1+bt['factor']).prod():.2f}x, B&H={(1+bt['bh']).prod():.2f}x")

Backtest Quality+LowVol: 127 mois, top-8, tx=10bps
Factor tilt: Sharpe=0.945, CAGR=25.43%
B&H equal:   Sharpe=0.967, CAGR=21.09%
Delta Sharpe: -0.022

Cumulative: Factor=11.00x, B&H=7.58x


## 3. Walk-forward 5-fold + multi-seed

Validation robuste : walk-forward 5-fold expanding avec 4 seeds.
Le seed injecte du bruit dans le score (jitter gaussien) pour tester la robustesse du signal.

In [4]:
print(f"{'='*70}")
print(f"Walk-forward 5-fold")
print(f"{'='*70}")

n = len(bt)
fold_size = n // (N_SPLITS + 1)

wf_results = []
for k in range(1, N_SPLITS + 1):
    tr_end = fold_size * k
    te_end = min(tr_end + fold_size, n)
    if te_end - tr_end < 10:
        continue
    
    # Use bt data directly (already computed factor scores on full history)
    # For walk-forward, we take the pre-computed returns and split
    test_slice = bt.iloc[tr_end:te_end]
    f_sharpe = sharpe_ann(test_slice['factor'])
    b_sharpe = sharpe_ann(test_slice['bh'])
    wf_results.append({
        'fold': k, 'n_oos': len(test_slice),
        'factor_sharpe': f_sharpe, 'bh_sharpe': b_sharpe,
        'delta': f_sharpe - b_sharpe
    })

wf_df = pd.DataFrame(wf_results)
print(wf_df[['fold', 'n_oos', 'factor_sharpe', 'bh_sharpe', 'delta']].to_string(index=False))
print(f"\nMean OOS Sharpe:")
print(f"  Factor tilt: {wf_df['factor_sharpe'].mean():.3f}")
print(f"  B&H equal:   {wf_df['bh_sharpe'].mean():.3f}")
print(f"  Delta:       {wf_df['delta'].mean():+.3f}")

# Multi-seed: add noise to scores and re-run
print(f"\n{'='*60}")
print(f"Multi-seed robustness ({len(SEEDS)} seeds)")
print(f"{'='*60}")

seed_results = []
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    # Add noise to composite scores
    noisy_scores = scores.copy()
    for col in noisy_scores.columns:
        noise = rng.normal(0, 0.5, len(noisy_scores))  # z-score noise
        noisy_scores[col] = noisy_scores[col] + noise
    
    try:
        bt_seed = run_factor_tilt_backtest(prices, noisy_scores.dropna(how='all'),
                                           n_long=N_LONG, tx_cost=TX_COST)
        if len(bt_seed) < 12:
            continue
        s_sharpe = sharpe_ann(bt_seed['factor'])
        s_cagr = (1 + bt_seed['factor']).prod() ** (12/len(bt_seed)) - 1
        max_dd = ((1+bt_seed['factor']).cumprod() / (1+bt_seed['factor']).cumprod().cummax() - 1).min()
        seed_results.append({
            'seed': seed, 'sharpe': s_sharpe, 'cagr': s_cagr, 'max_dd': max_dd
        })
    except Exception as e:
        print(f"  Seed {seed}: ERROR {e}")

seed_df = pd.DataFrame(seed_results)
if len(seed_df) > 0:
    print(seed_df[['seed', 'sharpe', 'cagr', 'max_dd']].to_string(index=False))
    print(f"\nStd Sharpe: {seed_df['sharpe'].std():.4f}")
    print(f"Sharpe original: {factor_sharpe:.4f}")
    print(f"Sharpe moyen bruite: {seed_df['sharpe'].mean():.4f}")

Walk-forward 5-fold
 fold  n_oos  factor_sharpe  bh_sharpe     delta
    1     21       0.640845   0.861770 -0.220925
    2     21       1.107810   1.166909 -0.059100
    3     21       0.682636   0.328944  0.353693
    4     21       1.014054   1.359021 -0.344967
    5     21       1.030027   1.159971 -0.129944

Mean OOS Sharpe:
  Factor tilt: 0.895
  B&H equal:   0.975
  Delta:       -0.080

Multi-seed robustness (4 seeds)


 seed   sharpe     cagr    max_dd
    0 0.947965 0.253677 -0.298487
    1 0.932791 0.249963 -0.392325
    7 1.040822 0.276956 -0.355309
   42 0.844029 0.219050 -0.478400

Std Sharpe: 0.0806
Sharpe original: 0.9451
Sharpe moyen bruite: 0.9414


## 4. Analyse par régime de marché

On classifie chaque mois en régime Bull/Neutral/Bear selon le rendement SPY sur 6 mois.

In [5]:
# Regime analysis using SPY 6-month momentum
spy = prices['SPY'].dropna()
spy_6m = spy.pct_change(126).resample('ME').last().dropna()

def classify_regime(ret):
    if ret > 0.10:
        return 'Bull'
    elif ret < -0.10:
        return 'Bear'
    return 'Neutral'

regimes = spy_6m.apply(classify_regime)
# Convert regimes index to PeriodIndex for alignment
regimes_period = regimes.copy()
regimes_period.index = regimes_period.index.to_period('M')

print(f"{'='*60}")
print(f"Performance par regime de marche")
print(f"{'='*60}")

# Align regime with backtest dates using PeriodIndex
bt_period_idx = bt.index.to_period('M')

for regime in ['Bull', 'Neutral', 'Bear']:
    mask = [regimes_period.get(p, None) == regime for p in bt_period_idx]
    mask = np.array(mask)
    n_months = mask.sum()
    if n_months < 3:
        print(f"{regime:8s}: {n_months} mois (insuffisant)")
        continue
    f_sh = sharpe_ann(bt['factor'][mask])
    f_cagr = (1+bt['factor'][mask]).prod() ** (12/n_months) - 1
    f_vol = bt['factor'][mask].std() * 12**0.5
    print(f"{regime:8s}: {n_months:3d} mois, Sharpe {f_sh:+.3f}, CAGR {f_cagr:+.2%}, Vol {f_vol:.2%}")

print(f"\n--- B&H equal par regime ---")
for regime in ['Bull', 'Neutral', 'Bear']:
    mask = [regimes_period.get(p, None) == regime for p in bt_period_idx]
    mask = np.array(mask)
    n_months = mask.sum()
    if n_months < 3:
        continue
    b_cagr = (1+bt['bh'][mask]).prod() ** (12/n_months) - 1
    print(f"{regime:8s}: {n_months:3d} mois, CAGR {b_cagr:+.2%}")

Performance par regime de marche
Bull    :  49 mois, Sharpe +1.705, CAGR +70.64%, Vol 35.36%
Neutral :  74 mois, Sharpe +0.528, CAGR +8.83%, Vol 19.62%
Bear    :   4 mois, Sharpe -6.014, CAGR -60.09%, Vol 14.56%

--- B&H equal par regime ---
Bull    :  49 mois, CAGR +54.94%
Neutral :  74 mois, CAGR +11.20%
Bear    :   4 mois, CAGR -71.45%


## 5. Verdict et conclusion

Le verdict est établi selon les critères de l'EPIC #1409 :
- **BEATS** : delta Sharpe ≥ +0.1 sur la majorité des folds OOS, edge ≥ 2σ cross-seed
- **NO BEATS** : delta Sharpe négatif ou edge < 2σ
- **INCONCLUSIVE** : résultats mixtes sans signal clair

In [6]:
# Verdict computation
delta_sharpe_full = factor_sharpe - bh_sharpe
mean_wf_delta = wf_df['delta'].mean()
n_positive_folds = (wf_df['delta'] > 0).sum()

if len(seed_df) >= 2:
    seed_deltas = seed_df['sharpe'].values - bh_sharpe
    mean_sd = seed_deltas.mean()
    std_sd = seed_deltas.std()
    sigma_edge = mean_sd / std_sd if std_sd > 1e-9 else float('nan')
else:
    sigma_edge = float('nan')

print(f"{'='*70}")
print(f"VERDICT — Strategy #5: Quality + LowVol Factor Tilt")
print(f"{'='*70}")
print(f"Factor Sharpe:     {factor_sharpe:.3f}")
print(f"B&H Sharpe:        {bh_sharpe:.3f}")
print(f"Delta Sharpe:      {delta_sharpe_full:+.3f}")
print(f"Walk-forward mean: {mean_wf_delta:+.3f} ({n_positive_folds}/{N_SPLITS} folds positive)")
if not np.isnan(sigma_edge):
    print(f"Cross-seed edge:   {sigma_edge:.2f} sigma")
else:
    print(f"Cross-seed edge:   N/A")

if delta_sharpe_full > 0.1 and n_positive_folds >= 3 and (np.isnan(sigma_edge) or sigma_edge >= 2.0):
    verdict = "BEATS"
elif delta_sharpe_full < 0 or n_positive_folds < 2:
    verdict = "NO BEATS"
else:
    verdict = "INCONCLUSIVE"

print(f"\n>>> VERDICT: {verdict} <<<")

# Scoreboard #35
print(f"\n{'='*70}")
print(f"Scoreboard #35 — 5 strategies")
print(f"{'='*70}")
scoreboard = [
    (1, 'Dual Momentum (L2)', 'NO BEATS', '#1575'),
    (2, 'Risk Parity', 'NO BEATS', '#1578'),
    (3, 'Trend Following (L1)', 'NO BEATS', '#1574'),
    (4, 'VRP Put-Write', 'NO BEATS', '#1579'),
    (5, 'Quality + LowVol', verdict, 'This PR'),
]
for n, name, v, pr in scoreboard:
    marker = ' <<<' if n == 5 else ''
    print(f"  {n}. {name:30s} | {v:15s} | {pr}{marker}")

VERDICT — Strategy #5: Quality + LowVol Factor Tilt
Factor Sharpe:     0.945
B&H Sharpe:        0.967
Delta Sharpe:      -0.022
Walk-forward mean: -0.080 (1/5 folds positive)
Cross-seed edge:   -0.37 sigma

>>> VERDICT: NO BEATS <<<

Scoreboard #35 — 5 strategies
  1. Dual Momentum (L2)             | NO BEATS        | #1575
  2. Risk Parity                    | NO BEATS        | #1578
  3. Trend Following (L1)           | NO BEATS        | #1574
  4. VRP Put-Write                  | NO BEATS        | #1579
  5. Quality + LowVol               | NO BEATS        | This PR <<<
